# 04 — Clasificador visual de brawlers
**Proyecto:** Recomendador de draft de Brawl Stars con reconocimiento visual de brawlers

**Objetivo:** entrenar un clasificador que, a partir del retrato de un brawler, identifique cuál es. Comparamos **KNN** contra **regresión logística** con características visuales clásicas (histograma de color y HOG).

**Diseño de evaluación (importante):** BrawlAPI entrega varias imágenes por brawler (`imageUrl`, `imageUrl2`, `imageUrl3`). Entrenamos con variaciones de `imageUrl` y evaluamos con variaciones de **otras imágenes del mismo brawler**. Así la prueba no reutiliza la misma imagen original. Si esas imágenes resultan idénticas, el notebook lo detecta y marca la evaluación como optimista.

## 1. Configuración

In [1]:
DATA_DIR = "/content/data"        # salida de los notebooks 02 y 03
IMG_SIZE = 64                     # tamaño para extraer características
AUG_TRAIN = 40                    # variaciones por brawler en entrenamiento
AUG_TEST = 15                     # variaciones por imagen de prueba
SEED = 42

## 2. Imports y catálogo de brawlers
Descargamos el catálogo de BrawlAPI y filtramos los 20 brawlers del alcance inicial. Los nombres se comparan en mayúsculas porque el dataset de partidas los usa así.

In [2]:
import io, hashlib, time
import numpy as np
import pandas as pd
import requests
from PIL import Image

np.random.seed(SEED)
rng = np.random.default_rng(SEED)

candidatos = pd.read_csv(f"{DATA_DIR}/selected_brawlers.csv")["brawler"].tolist()

r = requests.get("https://api.brawlapi.com/v1/brawlers", timeout=20)
r.raise_for_status()
catalogo = r.json()["list"]
por_nombre = {b["name"].upper(): b for b in catalogo}

faltantes = [n for n in candidatos if n not in por_nombre]
print("Brawlers seleccionados:", len(candidatos))
print("Sin coincidencia en BrawlAPI:", faltantes)
assert not faltantes, "Revisa los nombres (por ejemplo, 'MR. P' puede llamarse distinto en BrawlAPI)"

FileNotFoundError: [Errno 2] No such file or directory: '/content/data/selected_brawlers.csv'

## 3. Descarga de imágenes
Convertimos cada imagen a RGB sobre un fondo gris, porque los íconos PNG pueden tener transparencia. Guardamos también el hash de cada archivo para comprobar si las imágenes adicionales son realmente distintas.

In [ ]:
def bajar_imagen(url):
    if not url:
        return None, None
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()
    img = Image.open(io.BytesIO(resp.content)).convert("RGBA")
    fondo = Image.new("RGBA", img.size, (128, 128, 128, 255))
    fondo.alpha_composite(img)
    return fondo.convert("RGB"), hashlib.md5(resp.content).hexdigest()

fuentes = {}   # nombre -> lista de (fuente, imagen, hash)
for nombre in candidatos:
    b = por_nombre[nombre.upper()]
    lista = []
    for campo in ["imageUrl", "imageUrl2", "imageUrl3"]:
        img, h = bajar_imagen(b.get(campo))
        if img is not None:
            lista.append((campo, img, h))
    fuentes[nombre] = lista

resumen = pd.DataFrame([
    {"brawler": n, "imagenes": len(v), "hashes_unicos": len({h for _, _, h in v}),
     "tamano": v[0][1].size if v else None}
    for n, v in fuentes.items()
])
print("Imágenes distintas por brawler (hashes únicos):")
print(resumen["hashes_unicos"].value_counts().to_dict())
print("Tamaños encontrados:", resumen["tamano"].astype(str).value_counts().to_dict())
resumen.head(20)

## 4. Aumento de datos
Simulamos variaciones de captura: brillo, contraste, ruido y recorte ligero. Solo se aplican a imágenes de entrenamiento y de prueba, nunca se mezclan los conjuntos a nivel de imagen original.

In [ ]:
def aumentar(img, rng):
    a = np.asarray(img).astype(np.float32)
    a = a * rng.uniform(0.8, 1.2)                     # brillo
    m = a.mean()
    a = (a - m) * rng.uniform(0.8, 1.2) + m            # contraste
    a = a + rng.normal(0, 5, a.shape)                  # ruido
    im = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    w, h = im.size
    s = rng.uniform(0.9, 1.0)                          # recorte aleatorio
    cw, ch = int(w * s), int(h * s)
    x = rng.integers(0, w - cw + 1)
    y = rng.integers(0, h - ch + 1)
    return im.crop((x, y, x + cw, y + ch))

ejemplo = aumentar(fuentes[candidatos[0]][0][1], rng)
print("Tamaño original:", fuentes[candidatos[0]][0][1].size, "| tamaño aumentado:", ejemplo.size)

## 5. Características visuales
- **Histograma de color:** 8 bins por canal RGB, normalizado. Captura la paleta del brawler.
- **HOG:** descriptor de gradientes sobre la imagen en gris. Captura forma y contornos.

Concatenamos ambos vectores para cada imagen.

In [ ]:
from skimage.feature import hog

def caracteristicas(img):
    pequena = img.resize((IMG_SIZE, IMG_SIZE))
    arr = np.asarray(pequena)
    hist = np.concatenate([
        np.histogram(arr[..., c], bins=8, range=(0, 256))[0] for c in range(3)
    ]).astype(np.float32)
    hist /= hist.sum()
    gris = np.asarray(pequena.convert("L")) / 255.0
    descriptor = hog(gris, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2))
    return np.concatenate([hist, descriptor])

vector = caracteristicas(fuentes[candidatos[0]][0][1])
print("Dimensión del vector de características:", vector.shape)

## 6. Construcción de conjuntos de entrenamiento y prueba
- **Entrenamiento:** imagen original de `imageUrl` más `AUG_TRAIN` variaciones.
- **Prueba:** variaciones de `imageUrl2` e `imageUrl3`, solo si son distintas de `imageUrl`. Si no lo son, usamos variaciones de la imagen principal con otra semilla y marcamos la evaluación como optimista.

In [ ]:
X_train, y_train = [], []
X_test, y_test, origen_test = [], [], []

clases = candidatos.copy()
clase_idx = {n: i for i, n in enumerate(clases)}

hay_fuentes_distintas = 0
for nombre in clases:
    lista = fuentes[nombre]
    principal = next(img for campo, img, _ in lista if campo == "imageUrl")
    hash_principal = next(h for campo, _, h in lista if campo == "imageUrl")

    # Entrenamiento
    X_train.append(caracteristicas(principal)); y_train.append(clase_idx[nombre])
    for _ in range(AUG_TRAIN):
        X_train.append(caracteristicas(aumentar(principal, rng))); y_train.append(clase_idx[nombre])

    # Prueba: otras imágenes del mismo brawler
    otras = [(campo, img) for campo, img, h in lista if campo != "imageUrl" and h != hash_principal]
    if otras:
        hay_fuentes_distintas += 1
        for campo, img in otras:
            for _ in range(AUG_TEST):
                X_test.append(caracteristicas(aumentar(img, rng)))
                y_test.append(clase_idx[nombre]); origen_test.append(campo)
    else:
        for _ in range(AUG_TEST * 2):
            X_test.append(caracteristicas(aumentar(principal, rng)))
            y_test.append(clase_idx[nombre]); origen_test.append("misma_imagen")

X_train, y_train = np.array(X_train), np.array(y_train)
X_test, y_test = np.array(X_test), np.array(y_test)
evaluacion_optimista = hay_fuentes_distintas < len(clases)

print("Entrenamiento:", X_train.shape, "| clases:", len(np.unique(y_train)))
print("Prueba:", X_test.shape, "| origen:", pd.Series(origen_test).value_counts().to_dict())
print("Brawlers con imágenes adicionales distintas:", hay_fuentes_distintas, "de", len(clases))
if evaluacion_optimista:
    print("AVISO: evaluación optimista; algunas pruebas usan la misma imagen original.")

## 7. Entrenamiento y comparación
Entrenamos KNN (con k = 3) y regresión logística, ambos dentro de un pipeline que estandariza las características. Reportamos exactitud, precisión, exhaustividad y F1 macro, y comparamos con un clasificador de referencia que siempre predice la clase más frecuente.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

modelos = {
    "Referencia (clase mayoritaria)": make_pipeline(DummyClassifier(strategy="most_frequent")),
    "KNN (k=3)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
}

filas = []
ajustados = {}
for nombre, modelo in modelos.items():
    t0 = time.time()
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    p, r_, f1, _ = precision_recall_fscore_support(y_test, pred, average="macro", zero_division=0)
    filas.append({
        "modelo": nombre,
        "exactitud": round(accuracy_score(y_test, pred), 3),
        "precision_macro": round(p, 3),
        "exhaustividad_macro": round(r_, 3),
        "f1_macro": round(f1, 3),
        "tiempo_s": round(time.time() - t0, 2),
    })
    ajustados[nombre] = (modelo, pred)

tabla = pd.DataFrame(filas)
tabla

## 8. Matriz de confusión y errores
Revisamos qué brawlers se confunden con cuáles en el modelo de regresión logística. Las confusiones entre personajes de forma parecida son esperables y deben explicarse en el informe.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

_, pred_lr = ajustados["Regresión logística"]
cm = confusion_matrix(y_test, pred_lr, labels=range(len(clases)))

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(clases))); ax.set_yticks(range(len(clases)))
ax.set_xticklabels(clases, rotation=90, fontsize=8); ax.set_yticklabels(clases, fontsize=8)
ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
ax.set_title("Matriz de confusión: regresión logística")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

# Pares de confusión más frecuentes (fuera de la diagonal)
pares = [(clases[i], clases[j], cm[i, j]) for i in range(len(clases)) for j in range(len(clases)) if i != j and cm[i, j] > 0]
pares = sorted(pares, key=lambda x: -x[2])[:8]
print("Confusiones más frecuentes:")
for real, pred, n in pares:
    print(f"  {real} -> {pred}: {n}")

## 9. Guardado del modelo
Guardamos ambos pipelines con la lista de clases para usarlos en el notebook 05. Como las características dependen de `IMG_SIZE` y de la función `caracteristicas`, el notebook 05 debe usar exactamente la misma definición.

In [ ]:
import joblib, os

joblib.dump({
    "modelos": {k: v[0] for k, v in ajustados.items() if k != "Referencia (clase mayoritaria)"},
    "clases": clases,
    "img_size": IMG_SIZE,
    "evaluacion_optimista": evaluacion_optimista,
}, f"{DATA_DIR}/clasificador_visual.joblib")

tabla.to_csv(f"{DATA_DIR}/resultados_clasificador.csv", index=False)
print("Guardado:", [f for f in os.listdir(DATA_DIR) if "clasificador" in f or "resultados" in f])